# nb_mig_silver_20261116095000_CreateRegSubUnifiedTables_A

Create the various silver unified tables for Registration Submissions.

Note, since multiple developers are working on this simultaneously, the creation of tables has been divided across multiple migration notebooks

In [1]:
%run "/05 - shared-libraries/nb_shared_migrations" {"SUPPRESS_TABLE_EXISTS_EXCEPTIONS": true}

In [2]:
storage_account_name = "deintdlsaueaasqebla"

In [3]:
import os
import time
from pyspark.sql import SparkSession
from mdw_shared.migrations import ensure_history_table, is_applied, record_migration

spark = get_spark_session()

data_lake_paths = get_datalake_paths(storage_account_name)

METADATA_BASE_PATH = data_lake_paths["metadata"]
SILVER_BASE_PATH = data_lake_paths["silver"]

UNIFIED_MASTER_LOCATION_BASE_PATH = f"{SILVER_BASE_PATH}/unified/master"
UNIFIED_TRANSACTION_LOCATION_BASE_PATH = f"{SILVER_BASE_PATH}/unified/transaction/reg_sub"

MIGRATION_ID = "20261116095000_CreateRegSubUnifiedTables_A"
LAYER = "silver"
NOTEBOOK_NAME = "nb_mig_silver_20261116095000_CreateRegSubUnifiedTables_A"
ensure_history_table(spark, METADATA_BASE_PATH)

## Table Creation Functions

In [4]:
def _create_unified_master_table(table_name, cols):
    table_name = table_name.lower()
    location = f"{UNIFIED_MASTER_LOCATION_BASE_PATH}/{table_name}"
    create_silver_unified_master_table(table_name, location, cols)

def _create_unified_transaction_table(table_name, cols):
    table_name = table_name.lower()
    location = f"{UNIFIED_TRANSACTION_LOCATION_BASE_PATH}/{table_name}"
    create_silver_unified_master_table(table_name, location, cols)

In [5]:
def create_unified_master_tables():
    _create_unified_master_table("elicos_course", 
        cols = [
            ("elicos_course_id", "GUID"),
            ("name", "VARCHAR(MAX)"),
            ("created_by_id", "GUID"),
            ("created_by_name", "VARCHAR(MAX)"),
            ("created_by_yomi_name", "VARCHAR(MAX)"),
            ("created_on", "DATETIME"),
            ("created_on_behalf_by_id", "GUID"),
            ("created_on_behalf_by_name", "VARCHAR(MAX)"),
            ("created_on_behalf_by_yomi_name", "VARCHAR(MAX)"),
            ("import_sequence_number", "BIGINT"),
            ("modified_by_id", "GUID"),
            ("modified_by_name", "VARCHAR(MAX)"),
            ("modified_by_yomi_name", "VARCHAR(MAX)"),
            ("modified_on", "DATETIME"),
            ("modified_on_behalf_by_id", "GUID"),
            ("modified_on_behalf_by_name", "VARCHAR(MAX)"),
            ("modified_on_behalf_by_yomi_name", "VARCHAR(MAX)"),
            ("record_created_on", "DATETIME"),
            ("assigned_to_id", "GUID"),
            ("owner_id_name", "VARCHAR(MAX)"),
            ("owner_id_type_id", "STRING"),
            ("owner_id_yomi_name", "VARCHAR(MAX)"),
            ("owning_business_unit_id", "GUID"),
            ("owning_business_unit_name", "VARCHAR(MAX)"),
            ("owning_team_id", "GUID"),
            ("owning_user_id", "GUID"),
            ("status", "BIGINT"),
            ("status_reason", "BIGINT"),
            ("time_zone_rule_version_number", "BIGINT"),
            ("utc_conversion_time_zone_code", "BIGINT"),
            ("version_number", "BIGINT")
       ])

In [7]:
def create_unified_transaction_tables():
    return
    # _create_unified_transaction_table("", 
    #     cols = [

    #    ])

## Execute Migration

In [8]:
if is_applied(spark, METADATA_BASE_PATH, MIGRATION_ID):
    print(f"Already applied: {MIGRATION_ID}")
else:
    t0 = time.time()

    # 1. Create Tables
    create_unified_master_tables()
    create_unified_transaction_tables()

    # 2. Record Migration
    duration = time.time() - t0
    
    # TODO: uncomment this before PR
    #record_migration(spark, METADATA_BASE_PATH, MIGRATION_ID, LAYER, NOTEBOOK_NAME,"succeeded", duration_seconds=duration)
                     
    print(f"Applied: {MIGRATION_ID} in {duration:.1f}s")
